# Rust 110: Structs, OOP and Traits

Chapter 10 of the Rust track, and the one where you finally build your own types. Chapters 01–09 used the standard library's structs and traits as a client — `Vec`, `String`, `HashMap`, `Option`, `Display`, `Iterator` — and chapter 08's macros kept demanding `Debug` and `PartialEq` you could only *derive*. This chapter crosses to the other side of the line: defining structs and enums, attaching behaviour with `impl`, sharing contracts with traits, and making types that work across any type with generics. It is also the applied track's opening chapter — everything from here on builds services out of types you define yourself.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `10_structs-oop-and-traits_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer, arriving from Python and C++. You have written classes with methods, interfaces, and templates or generics before — none of that is new. What is new: behaviour in Rust is attached by *impl blocks* rather than declared inside the type; polymorphism is a **choice between two dispatch strategies** (static, compiled per type, or dynamic, through a vtable) rather than a single inheritance mechanism; and `+` or `==` on your own types is not magic — it is a trait, the same kind of contract you already write yourself.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Cells are independent — there is no sandbox and no shared state beyond the kernel namespace, so a restart-and-run-from-the-top always works.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic or panic message you would get. §13 is the catalogue.
- One cell (§12.2) uses `#[repr(C)]` purely to measure layout; every other cell is plain Rust.
- The prose is terse and the facts are dense. This is a reference you will come back to, not a novel you read once.

## The one idea to internalise

**Data is dumb, behaviour is bolted on, and contracts are traits.** A `struct` holds fields and nothing else; an `impl` block attaches functions to it, and nothing in the block is visible unless you say `pub`; a `trait` names a set of capabilities that any type can promise. This separation — Python bolts methods onto the class, C++ declares them inside it, Rust keeps all three apart — is what lets the same `Display` contract serve your types and the standard library's alike, and it is why `Order::new` and `order.notional()` are both just functions with a `Self` in scope. The chapter's habit: when you design a type, design the *trait* it satisfies first — the struct is just one implementation of it.


### Contents

- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. Struct Anatomy: Three Forms and Their Constructors](#1-struct-anatomy-three-forms-and-their-constructors)
- [2. Impls: Associated Functions, Methods and `Self`](#2-impls-associated-functions-methods-and-self)
- [3. Visibility: `pub` and the Privacy Boundary](#3-visibility-pub-and-the-privacy-boundary)
- [4. Derive: Free Impls from One Line](#4-derive-free-impls-from-one-line)
- [5. Newtypes: Distinct Types at Zero Cost](#5-newtypes-distinct-types-at-zero-cost)
- [6. Traits: Contracts with Default Methods](#6-traits-contracts-with-default-methods)
- [7. Generics: One Definition, Many Types](#7-generics-one-definition-many-types)
- [8. Trait Objects: `dyn` and Dynamic Dispatch](#8-trait-objects-dyn-and-dynamic-dispatch)
- [9. Enums with Data: The Sum Type](#9-enums-with-data-the-sum-type)
- [10. Operators Are Traits: `Add`, `Display` and `Debug`](#10-operators-are-traits-add-display-and-debug)
- [11. Static vs Dynamic Dispatch: The Cost Model](#11-static-vs-dynamic-dispatch-the-cost-model)
- [12. Layout: Padding, Field Order and `repr`](#12-layout-padding-field-order-and-repr)
- [13. Reading the Compiler: Panics and E-Codes](#13-reading-the-compiler-panics-and-e-codes)
- [14. Mental Model: The Three-Way Split](#14-mental-model-the-three-way-split)
- [15. Cheat Sheet and Review](#15-cheat-sheet-and-review)

If you are returning to this chapter later, §15 is the one-page summary, §13 is the diagnostics catalogue, and §11 is where the dispatch cost numbers live.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. Three consequences matter for every cell below:

- **There is no `fn main`.** The kernel supplies one; you are typing its body.
- **State persists across cells.** A `let` from cell 4 is still live in cell 20 — including `use` statements.
- **Items are order-independent, statements are not.** Two `fn`s may call each other regardless of order; two `let`s may not use each other's names before they run.

There is **one namespace per notebook**. Two items with the same name collide exactly as they would in one source file, even in different sections. This chapter *defines types*, so the rule gets sharper: every `struct`, `enum`, `trait` and `impl` name below is unique across the whole notebook. If you re-run a cell that defines an item, it is redefined; on a redefinition error, restart the kernel and run from the top.

A failing cell does not poison the kernel — the deliberate compile errors in this chapter are comments, so every cell still runs.

### A chapter without a sandbox

Unlike chapter 07, this notebook touches **no files**: defining types, implementing traits and measuring layout are in-process by nature. That is itself a lesson — everything here is about the shape of your program, not the world outside it.


In [ ]:
// 0.1 Smoke test: a tuple struct, a method, a trait bound — the chapter in one line.
struct BasisPoints(u32);
impl BasisPoints {
    fn as_fraction(self) -> f64 {
        self.0 as f64 / 10_000.0
    }
}
let spread = BasisPoints(12).as_fraction();
println!("kernel alive, 12 bp = {spread}");
assert_eq!(BasisPoints(12).as_fraction(), 0.0012);


## 1. Struct Anatomy: Three Forms and Their Constructors

A **struct** is a named product of named parts. Rust has three forms, and the difference is how you reach the fields:

| Form | Written as | Fields live as | Use for |
|---|---|---|---|
| named-field | `struct Order { qty: u32, px: f64 }` | `o.qty`, `o.px` | the default — data with meaningful names |
| tuple struct | `struct Qty(u32);` | `self.0` | newtypes; the name *is* the meaning |
| unit struct | `struct Closed;` | (none) | a zero-cost type-level marker |

The zero-size fact is not cosmetic: `size_of::<Closed>() == 0`, verified in §12.1, so a unit struct in another type costs nothing — it is Rust's answer to a marker interface or a `bool` flag that is really a type decision.

Construction is by **struct literal** — every field, no half-built values:

### Field-init shorthand and functional update


In [ ]:
// 1.1 The three construction moves: literal, field-init shorthand, functional update.
#[derive(Debug)]
struct Order {
    symbol: String,
    qty: u32,
    px_cents: u32,
}
let o = Order { symbol: String::from("AAPL"), qty: 300, px_cents: 19_251 };
// field-init shorthand: a local named like the field populates it
let symbol = String::from("MSFT");
let o2 = Order { symbol, qty: 100, px_cents: 41_500 };
// functional update `..base`: fields not listed come from the base expression
let o3 = Order { qty: 7, ..o2 };
println!("{o3:?}");
assert_eq!(o3.symbol, "MSFT");
assert_eq!(o.qty, 300);
assert_eq!(o.px_cents, 19_251); // explicit read: Debug renders don't count as reads


Two rules govern that cell. **Field-init shorthand** copies a *binding* into a same-named field — it needs a local variable, not an expression name. **Functional update** `..base` moves the un-listed fields out of the base: `o3` took `symbol` and `px_cents` from `o2`, leaving `o2` partially moved — §1.2 measures exactly which fields survive and which diagnostic fires on the rest.

Constructors are ordinary associated functions — there is no special `__init__` and no constructor overload table; the convention is one `new` plus builder-style `with_*` helpers (§2). A missing field is a compile error, not a default:

```text
error[E0063]: missing field `symbol` in initializer of `Order`
```

and a fieldful enum cast to its discriminant is refused outright (§9): the compiler compiles the struct literal *completely* or not at all.

### The partial-move edge of `..base`


In [ ]:
// 1.2 Functional update moves non-Copy fields out of the base.
#[derive(Debug)]
struct Tiny { a: u32, b: String }
let t = Tiny { a: 1, b: String::from("keep") };
let t2 = Tiny { a: 9, ..t };
println!("{t2:?}");
println!("t.a survives: {}", t.a);          // fine: u32 is Copy, and `a` was re-specified
// println!("{}", t.b); // error[E0382]: borrow of moved value: `t.b`
assert_eq!(t2.b, "keep");


`..t` moved `t.b` (a `String` — not `Copy`) into `t2`, so `t` is *partially moved*: still alive, still readable field-by-field for `Copy` fields like `t.a`, but its `String` is gone. Reading `t.b` is E0382, the same diagnostic chapter 05 catalogued. The habit: functional update is a convenience for re-specifying a few fields, not a clone — say `.clone()` explicitly when you want the base kept whole.

## 2. Impls: Associated Functions, Methods and `Self`

An **impl block** attaches functions to a type. Two shapes live inside it, and the receiver is the whole difference:

| Shape | First parameter | Called as | Good for |
|---|---|---|---|
| **associated function** | none | `Order::new(..)` | constructors, parsing, `Default`-style values |
| **method** | `self` in some form | `order.notional()` | behaviour on an instance |

`Self` (capital) is the type being impl'd — usable as a return type, as `Self { .. }` in a constructor — while `self` (lowercase) is the receiver argument, and it comes in the same three flavours chapter 05 taught for plain functions: `&self` borrows, `&mut self` mutates, `self` consumes. The compiler's rule for *writing* the receiver: **method calls auto-ref**. `t.bump(0.5)` on a non-reference binding compiles as `(&mut t).bump(0.5)`; you never write `(&mut t).bump` by hand.

### Constructors, methods, and the consuming receiver


In [ ]:
// 2.1 Associated functions, methods on all three receivers.
#[derive(Debug)]
struct Ticket {
    symbol: String,
    qty: u32,
}
impl Ticket {
    /// Associated function: no receiver — a constructor.
    fn new(symbol: &str, qty: u32) -> Self {
        Ticket { symbol: symbol.to_string(), qty }
    }
    /// &self: read-only borrow.
    fn notional(&self, px: f64) -> f64 {
        self.qty as f64 * px
    }
    /// &mut self: in-place update.
    fn scale(&mut self, k: u32) {
        self.qty *= k;
    }
    /// self: consumes the value, field-by-field.
    fn into_parts(self) -> (String, u32) {
        (self.symbol, self.qty)
    }
}
let mut t = Ticket::new("ES", 10);
println!("{}", t.notional(4126.5));
t.scale(3);
assert_eq!(t.qty, 30);
let (symbol, qty) = t.into_parts();   // t is consumed here
println!("{symbol} x{qty}");
// println!("{}", t.qty); // error[E0382]: borrow of moved value: `t`


Three facts hide in that cell. First, **an instance method is exactly a function with `Self` as its first parameter** — `t.notional(4126.5)` and `Ticket::notional(&t, 4126.5)` are the same call; the dot form is sugar for passing the receiver. That equivalence is what §10's UFCS exploits when two traits define the same method name. Second, **`self` by value consumes** — `into_parts` took `t` apart field-by-field, which is why the print of `t.qty` afterwards is commented E0382. Third, **multiple `impl` blocks on one type are fine** — the block is a grouping, not a declaration, so you can spread impls across a file or gate one behind a `cfg` (chapter 13).

A method may share a name with a field — `fn qty(&self)` next to field `qty` compiles, and `t.qty` is the field while `t.qty()` is the method. It is legal and occasionally good API design; the auto-ref rules untangle which is meant by the parentheses.

### The field/method shadow, and one namespace per type


In [ ]:
// 2.2 A method named like a field: both are callable, the parens decide.
#[derive(Debug)]
struct Holding {
    qty: u32,
    symbol: String,
}
impl Holding {
    fn qty(&self) -> u32 {
        self.qty
    }
    fn label(&self) -> String {
        format!("{} x{}", self.symbol, self.qty)
    }
}
let f = Holding { qty: 100, symbol: String::from("ES") };
let field = f.qty;
let method = f.qty();
println!("field {field}, method {method}, label {}", f.label());
assert_eq!(field, method);


## 3. Visibility: `pub` and the Privacy Boundary

Rust's default visibility is **private to the module** — and in this notebook, the whole notebook is one module, so every type here is freely visible to every other cell. That makes privacy one of those facts this notebook can state but not *demonstrate*: a `pub` boundary only bites when a second module exists (chapter 13's subject). The rules to carry forward now:

| Item | Default | `pub` grants |
|---|---|---|
| struct field | private to the module | others can read *and construct* with it |
| enum variant | as public as the enum | no per-variant privacy switch |
| trait method | as public as the trait | no per-method privacy switch |
| `impl` method | private to the module | callable from other modules |

The design consequence is the **constructor-and-getters pattern**: a struct with private fields plus `new` and accessor methods makes illegal states unrepresentable — outside code cannot build one with `qty: 0` if the only constructor refuses it. Python reaches for `@property` and name-mangling; C++ reaches for `private:` sections; Rust reaches for *not writing* `pub` on the field.

### The encapsulation idiom, stated in code


In [ ]:
// 3.1 Private fields + constructor + accessors: the encapsulation idiom.
// (In one notebook module everything is visible; the pattern is about API design.)
#[derive(Debug)]
struct Account {
    balance_cents: i64,   // would be invisible to other modules: no `pub`
}
impl Account {
    fn new() -> Self {
        Account { balance_cents: 0 }
    }
    fn deposit(&mut self, cents: i64) {
        assert!(cents > 0, "deposit must be positive");
        self.balance_cents += cents;
    }
    fn balance(&self) -> i64 {
        self.balance_cents
    }
}
let mut acct = Account::new();
acct.deposit(10_000);
acct.deposit(2_500);
println!("balance = {} cents", acct.balance());
assert_eq!(acct.balance(), 12_500);
// Account { balance_cents: 1_000_000 }; // in another module: private-field error


The commented-out literal is the pattern's whole point: *if* the field were visible outside the module, `Account { balance_cents: 1_000_000 }` would bypass `deposit`'s invariant check. Privacy is what makes the invariant an invariant. The `pub` keyword itself is one word, but its reach is per-item — `pub` on the struct without `pub` on the fields exports an opaque type, which is exactly the encapsulation idiom.

## 4. Derive: Free Impls from One Line

`#[derive(Debug, Clone, PartialEq)]` asks the compiler to write the impls for you, mechanically, from the fields. Each derivable trait has a precise meaning, and the derivations are *compositional*: a struct can derive `Debug` only if every field can.

| Trait | Derives | The derived behaviour |
|---|---|---|
| `Debug` | `{:?}` | one-line `{ Field: value }`; `{:#?}` pretty-prints multi-line |
| `Clone` | `.clone()` | field-wise deep copy (needs every field `Clone`) |
| `Copy` | implicit copies | **marker** — every field must be `Copy`; `String` blocks it |
| `PartialEq` | `==`, `!=` | field-wise comparison, in declaration order |
| `Eq` | marker | total equality (`HashMap`/`HashSet` keys need it) |
| `PartialOrd` | `<`, `>`, `<=`, `>=` | lexicographic, field order decides |
| `Ord` | sorting, `max` | total order (`BTreeMap` keys need it) |
| `Hash` | hashing | field-wise (`HashSet`/`HashMap` keys need it) |
| `Default` | `Default::default()` | `0`/`false`/`""`/`None` per field type |

Three order-of-operations facts worth pinning: `Copy` *implies* `Clone` (the compiler enforces `#[derive(Copy, Clone)]` together — deriving `Copy` without `Clone` is an error); `Eq` is a **marker with no methods** — it only tells generic code "this `PartialEq` never returns nonsense for `NaN`-like values"; and derived `PartialOrd` compares fields in **declaration order**, so field order is API — reordering `PartialOrd` fields changes every `<` in the codebase.

### The derive set, exercised


In [ ]:
// 4.1 One derive line, every behaviour: Debug, Clone, PartialEq, Hash, Default.
use std::collections::HashSet;
#[derive(Debug, Clone, PartialEq, Eq, Hash, Default)]
struct Price {
    symbol: String,
    cents: u32,
}
let p = Price { symbol: String::from("AAPL"), cents: 19_251 };
println!("debug: {p:?}");
println!("pretty: {p:#?}");
// Clone: the original survives
let p2 = p.clone();
assert_eq!(p, p2);
// Eq + Hash: usable as a set element
let mut seen = HashSet::new();
seen.insert(p.clone());
assert!(seen.contains(&p));
// Default: both fields get their type's default
let d = Price::default();
println!("default: {d:?}");
assert_eq!(d, Price { symbol: String::new(), cents: 0 });


Note what `Copy` did **not** appear in that derive line: `Price` contains a `String`, which is not `Copy`, so deriving `Copy` would be refused with an error naming the offending field. The next section turns that exclusion into a feature.

## 5. Newtypes: Distinct Types at Zero Cost

A **newtype** is a tuple struct with one field: `struct Qty(u32);`. The new type is *distinct* from its inner type — `Qty(300)` is not a `u32`, cannot be compared to one, cannot be passed where one is expected — yet it costs nothing: `size_of::<Qty>() == size_of::<u32>()` (§12.1). Chapter 01 promised exactly this when it said `type Celsius = f64` is *transparent*, and "to get a genuinely distinct type you need a newtype struct".

This is Rust's zero-cost answer to a whole shelf of Python/C++ patterns: a subclass of `int`, a `typedef` that wants to be stricter, an `enum class` with a payload, a units library. The compiler now tracks your unit for you — adding `Qty` to `BasisPoints` is a type error before it is a business error.

### The newtype, built out


In [ ]:
// 5.1 A newtype with a constructor invariant, operators, and a Display impl.
use std::fmt::{self, Display};
/// Quantity in lots; zero is illegal, so the constructor enforces it.
#[derive(Debug, Clone, Copy, PartialEq, Eq, PartialOrd, Ord)]
struct Qty(u32);
impl Qty {
    /// The single constructor: the type's invariant lives here.
    fn new(raw: u32) -> Option<Self> {
        if raw > 0 { Some(Qty(raw)) } else { None }
    }
    fn value(self) -> u32 {
        self.0
    }
}
impl Display for Qty {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "{} lots", self.0)
    }
}
let q = Qty::new(300).expect("300 is legal");
println!("q = {q}, inner = {}", q.value());
assert_eq!(q.value(), 300);
assert_eq!(size_of::<Qty>(), size_of::<u32>()); // zero-cost
assert!(Qty::new(0).is_none());                 // invariant enforced
// let _ = q + 1; // type error: Qty has no Add<u32> yet — that is §10's job


Read the derive line as a sentence: **a `Qty` is `Copy`, comparable for equality, orderable, printable, and hashable — and impossible to build with zero.** That last clause is the newtype's real power: the invariant is enforced at the *only* place a `Qty` can come into existence. Two more facts about newtypes before we move on:

- **Newtypes deref by hand, not automatically.** `q.value()` is a method we wrote; there is no implicit conversion to `u32`. (`Deref` exists — chapter 14's subject — and is deliberately not implicit for newtypes.)
- **`#[derive(Default)]` would give `Qty(0)`** — which violates the invariant. A newtype with a constrained constructor should implement `Default` by hand, or not at all.

### The marker: a unit struct as a zero-cost tag


In [ ]:
// 5.2 Unit structs as zero-cost type-level markers.
#[derive(Debug, Default)]
struct Closed;
/// A function only the marker's existence unlocks: the type IS the argument.
fn gate(_: Closed) -> &'static str {
    "market closed; orders rejected"
}
println!("{} (Closed is {} bytes)", gate(Closed::default()), size_of::<Closed>());
assert_eq!(size_of::<Closed>(), 0);


`Closed` occupies no memory and exists only so signatures can *require* it — `gate(_: Closed)` can only be called with a value only `Closed::default()` can supply. That is the type-system-as-assertion habit chapter 08 preached, now with a type you built.

## 6. Traits: Contracts with Default Methods

A **trait** declares a set of capabilities a type can promise. Where Python bolts methods onto the class and C++ declares them inside it, Rust's trait stands alone: `impl Trade for Summarize`-shaped contracts that *your* types and standard-library types both satisfy. The trait is the contract; the impl is the promise kept.

```rust
trait Summarize {
    fn headline(&self) -> String;
}
```

Anything with a `headline` method can now be used *through* `Summarize` — and a **default method** gives implementors a behaviour for free, overridable at will. Default methods are the Rust idiom for "the common case is provided; override only what differs."

### The contract, two implementations, one default


In [ ]:
// 6.1 Trait with a default method: Trade overrides summary, Halt does not.
trait Summarize {
    /// The one required method: the contract's core.
    fn headline(&self) -> String;
    /// Provided: every implementor gets this unless it opts out.
    fn summary(&self) -> String {
        format!("[feed] {}", self.headline())
    }
}
struct Trade {
    symbol: String,
    qty: u32,
}
impl Summarize for Trade {
    fn headline(&self) -> String {
        format!("{} x{}", self.symbol, self.qty)
    }
}
struct Halt {
    symbol: String,
}
impl Summarize for Halt {
    fn headline(&self) -> String {
        format!("HALT {0}", self.symbol)
    }
    fn summary(&self) -> String {
        format!("[risk] {}", self.headline()) // override
    }
}
let t = Trade { symbol: String::from("CL"), qty: 10 };
let h = Halt { symbol: String::from("NVDA") };
println!("{}", t.summary());
println!("{}", h.summary());
assert_eq!(t.summary(), "[feed] CL x10");
assert_eq!(h.summary(), "[risk] HALT NVDA");


The default method's body can call `self.headline()` — the *required* method — which is the standard shape: defaults build on requirements, never the reverse. Overriding is implicit (re-declare the method in the impl); "calling the parent" has no `super` sugar — if a default body matters, factor the shared logic into a free function or a second required method.

**Associated consts** live in traits too: `const SOURCE: &str` with a per-impl value. They come with a catch that the probe caught honestly: a trait containing an associated const is **not dyn-compatible** — `Box<dyn Summarize>` with a const member fails E0038 (§8, §13). The practical rule: keep constants out of traits you intend to use as trait objects.

### Two traits, one method name: UFCS


In [ ]:
// 6.2 Same method name from two traits: only fully qualified syntax disambiguates.
trait Named {
    fn describe(&self) -> String;
}
trait Priced {
    fn describe(&self) -> String;
}
struct Quote {
    symbol: String,
    cents: u32,
}
impl Named for Quote {
    fn describe(&self) -> String {
        format!("named {0}", self.symbol)
    }
}
impl Priced for Quote {
    fn describe(&self) -> String {
        format!("priced {0}", self.cents)
    }
}
let q = Quote { symbol: String::from("AAPL"), cents: 19_251 };
// q.describe(); // error[E0034]: multiple applicable items in scope
println!("{}", <Quote as Named>::describe(&q));
println!("{}", <Quote as Priced>::describe(&q));
assert_eq!(<Quote as Named>::describe(&q), "named AAPL");
assert_eq!(<Quote as Priced>::describe(&q), "priced 19251");


`<Quote as Named>::describe(&q)` is **fully qualified syntax**: "call `Named`'s `describe` on `q`." You need it exactly when ambiguity exists — two trait methods with one name, or a trait method shadowed by an inherent one. Everywhere else the dot form wins on readability; write the qualified form only when the compiler demands it (E0034) or when naming the trait is the point.

## 7. Generics: One Definition, Many Types

A **generic** is a type parameter: `struct Candle<T> { open: T, close: T }` is not one type but a *family*, stamped out per concrete `T` the code touches. Chapter 06's `Vec<i32>` was already this — `Vec` is generic, `Vec<i32>` is one stamping. The mechanics that matter for designing your own:

| Syntax | Meaning | Where it bites |
|---|---|---|
| `struct Candle<T>` | the family name | `Candle` alone is not a type; `Candle<f64>` is |
| `impl<T> Candle<T>` | impl for every `T` | methods available only where bounds allow |
| `T: PartialOrd` | a **bound**: T must do these things | unbounded `T` can do nothing but drop |
| `where T: A + B` | bounds spelled below, for complex signatures | preferred in real code |
| `impl Trait` in arg | "some one type implementing Trait" | sugar for a generic parameter |
| `-> impl Trait` | "some one type, hidden" | **one** concrete type per function body |

The bound is the load-bearing part: a bare `T` admits every type but lets the body do *nothing* with it except move it around. Bounds are what make the body's operations compile — and they are checked at the *call site* too: `Candle::<String>::mid()` refuses to exist because `String` is not `Copy`.

### Bounds and the where form


In [ ]:
// 7.1 Generic struct, bounded impl, where clause, turbofish.
#[derive(Debug)]
struct Candle<T> {
    open: T,
    close: T,
}
impl<T> Candle<T>
where
    T: Copy + PartialOrd,
{
    /// Returns (low, high); needs T: Copy to return the values, PartialOrd to compare.
    fn range(&self) -> (T, T) {
        if self.open <= self.close {
            (self.open, self.close)
        } else {
            (self.close, self.open)
        }
    }
}
let c = Candle { open: 4125.0f64, close: 4126.5 };
println!("{c:?} range {:?}", c.range());
assert_eq!(c.range(), (4125.0, 4126.5));
// Candle::<String>::range would not exist: String is neither Copy nor comparable.
// The turbofish form names T explicitly at the call site:
let ci = Candle::<u32> { open: 10, close: 20 };
assert_eq!(ci.range(), (10, 20));


The compiler refuses the method, not the struct: `Candle<String>` *exists*, it just has no `range` — bounds gate **impl blocks**, not type constructors. That is the standard shape: data structures generic over everything, behaviour unlocked by bounds.

`-> impl Trait` has the sharpest edge in the chapter: the function's *body* must resolve the hidden type to **one concrete type**, decided at compile time. Two arms returning different types under `if`/`else` is:

```text
error[E0308]: `if` and `else` have incompatible types
```

— reported at the branch, but caused by the signature's one-hidden-type promise. To return different types by a runtime flag, you need a trait object (next section).

### The one-hidden-type rule


In [ ]:
// 7.2 impl Trait in argument position: sugar for a generic; body sees the bound only.
fn loudest(items: &[impl Summarize]) -> Option<String> {
    items.iter().map(|i| i.summary()).max_by_key(|s| s.len())
}
let trades = [
    Trade { symbol: String::from("CL"), qty: 10 },
    Trade { symbol: String::from("HO"), qty: 5 },
];
println!("loudest: {:?}", loudest(&trades));
assert_eq!(loudest(&trades).as_deref(), Some("[feed] CL x10"));
let empty: Vec<Trade> = Vec::new();
assert_eq!(loudest(&empty).as_deref(), None);


`impl Trait` in argument position *is* a generic parameter with an elided name — `loudest` compiles to one monomorphised copy per concrete `T` used. (The `Summarize` trait and its impls come from §6 — items are order-independent, and the validator's hoisting makes that explicit.)

## 8. Trait Objects: `dyn` and Dynamic Dispatch

Generics choose the type at compile time. A **trait object** defers the choice to run time: `Box<dyn Report>` is a box that holds *some* `Report`, decided when the box was built, dispatching through a **vtable** — a per-type table of function pointers the compiler laid out. The trade is §11's subject; here, the mechanics and the rules.

| Form | Meaning | Size |
|---|---|---|
| `&dyn Trait` / `&mut dyn Trait` | borrowed, fat pointer (data ptr + vtable ptr) | 16 bytes |
| `Box<dyn Trait>` | owned, same fat pointer inside the box | 8 bytes of handle, data on heap |
| `Rc<dyn Trait>` | shared ownership of some trait object | chapter 14 |

A `dyn` reference is a **fat pointer**: pointer to the data, pointer to the type's vtable. That is why `size_of::<&dyn Trait>() == 16` while `size_of::<&Concrete>() == 8` — the vtable pointer is the second word, and it is what makes `mixed[0].headline()` call the *right* function without knowing the type.

### The vtable, exercised


In [ ]:
// 8.1 Box<dyn Trait>: one vector, two types, one contract.
/// A dyn-compatible twin of §6's Summarize: no consts, no generics, no `self`-by-value.
/// (Its method is `flash`, not `headline`: Halt already gets `headline` from
/// Summarize, and a second trait method of the same name would make every
/// Halt receiver ambiguous — E0034, §6.2's disambiguation problem.)
trait Report {
    fn flash(&self) -> String;
}
impl Report for Trade {
    fn flash(&self) -> String {
        format!("{} x{}", self.symbol, self.qty)
    }
}
impl Report for Halt {
    fn flash(&self) -> String {
        format!("HALT {0}", self.symbol)
    }
}
let mixed: Vec<Box<dyn Report>> = vec![
    Box::new(Trade { symbol: String::from("GC"), qty: 1 }),
    Box::new(Halt { symbol: String::from("NVDA") }),
];
for m in &mixed {
    println!("dyn: {}", m.flash()); // vtable dispatch: the right impl per element
}
assert_eq!(mixed[0].flash(), "GC x1");
assert_eq!(mixed[1].flash(), "HALT NVDA");
assert_eq!(size_of::<&dyn Report>(), 16); // fat pointer


The loop's `m.flash()` is the entire feature: one call site, two implementations, the vtable picks. Nothing about `Trade` or `Halt` appears in this cell — the objects carry their own behaviour, which is the textbook definition of runtime polymorphism. The costs and the compile-time alternative are §11's numbers.

### The dyn-compatibility rules


In [ ]:
// 8.2 What disqualifies a trait from dyn: probe-backed catalogue.
// A trait is "dyn compatible" (older term: "object safe") when a vtable can exist for it.
// The blockers, each verified by this chapter's probes:
//
// - associated consts          -> E0038 (§6's Summarize has one)
// - generic methods            -> E0038 (fn eq_other<T>(&self, t: T))
// - methods taking self by value -> E0161/E0507 at the call site (&dyn Eater)
//
trait Flex {
    fn show(&self) -> String {
        "flex".to_string()
    }
    // fn eq_other<T>(&self, other: T) -> bool { false } // <- would block dyn Flex
}
trait Eater {
    fn eat(self) -> u32; // <- blocks dyn only when called *through* a &dyn
}
struct Flexy;
impl Flex for Flexy {}
struct Meal;
impl Eater for Meal {
    fn eat(self) -> u32 {
        1
    }
}
let f = Flexy;
println!("direct call fine: {}", f.show());
let m = Meal;
println!("direct by-value call fine: {}", m.eat());
// let d: &dyn Eater = &Meal;
// d.eat(); // error[E0161]: cannot move a value of type `dyn Eater`
//          // error[E0507]: cannot move out of `*d` which is behind a shared reference


The rules exist for one reason: a vtable entry must be a plain function pointer. A generic method needs a *different* entry per `T` — no single pointer to store. A by-value-`self` method needs to know the size it is moving; `dyn Trait` erases exactly that. The compiler's umbrella diagnostic is **E0038**, and its help text suggests the standard fixes: replace the associated const with a required method, or replace the generic method with one taking `&dyn` arguments.

## 9. Enums with Data: The Sum Type

A Rust `enum` is not C's `enum` — it is a **sum type**: each variant may carry its own data, and the enum holds exactly one variant at a time. You have used them as a client since chapter 05 (`Option`, `Result`); this section builds them as an author. The shape:

```rust
enum Fill {
    New,
    Partial { qty: u32, remaining: u32 },
    Full { qty: u32 },
}
```

Three variants, three shapes: nothing, two named fields, one named field. Methods attach with `impl` exactly as on structs, `match` destructures (chapter 02), and `#[derive(Default)]` + `#[default]` names the zero value.

### The sum type, built and destructured


In [ ]:
// 9.1 Enum with data: methods, #[default], match destructuring.
#[derive(Debug, Default, PartialEq)]
enum Fill {
    #[default]
    New,
    Partial { qty: u32, remaining: u32 },
    Full { qty: u32 },
}
impl Fill {
    /// Filled quantity so far, whatever the variant.
    fn filled(&self) -> u32 {
        match self {
            Fill::New => 0,
            Fill::Partial { qty, .. } => *qty,
            Fill::Full { qty } => *qty,
        }
    }
}
let fills = [
    Fill::Partial { qty: 300, remaining: 120 },
    Fill::Full { qty: 300 },
    Fill::New,
];
for f in &fills {
    println!("{f:?} filled={}", f.filled());
}
assert_eq!(fills[0].filled(), 300);
assert_eq!(Fill::default(), Fill::New);
assert!(matches!(fills[1], Fill::Full { .. }));


`matches!` (chapter 08) slots straight in on top of your own enum — `assert!(matches!(fills[1], Fill::Full { .. }))` asserts the *shape*, not a re-derived accessor. Note `Fill::Partial { qty, .. }`'s rest pattern: it ignores `remaining` where `filled()` does not need it.

Fieldful variants change one rule from chapter 01's fieldless enums: **casting to the discriminant is refused** — `Fill::New as u8` is:

```text
error[E0605]: non-primitive cast: `Fill` as `u8`
```

because with data aboard there is no single integer to cast. Fieldless enums keep the cast; with **explicit discriminants** they become typed constants:

### Fieldless enums and explicit discriminants


In [ ]:
// 9.2 Fieldless enum: explicit discriminants, as-cast, derived ordering.
#[derive(Debug, PartialEq, Eq, PartialOrd, Ord)]
enum Level {
    Low = 1,
    Mid = 5,
    High = 10,
}
assert_eq!(Level::Mid as u8, 5);
assert!(Level::Low < Level::High);
// PartialOrd on an enum: declaration order decides — Low < Mid < High here,
// regardless of the discriminant values.
let ranked = [Level::High, Level::Low];
let mut sorted = ranked;
sorted.sort();
assert_eq!(sorted, [Level::Low, Level::High]);


Two orderings live on one enum and they are **not** the same: `as` reads the *discriminant* (1/5/10), while derived `Ord` uses *declaration order* (Low < Mid < High — which here agrees, but rewrite `Low = 10` and they would disagree). The design rule: discriminants are for wire formats and FFI; declaration order is for `Ord`. Keep both aligned or drop one.

## 10. Operators Are Traits: `Add`, `Display` and `Debug`

Chapter 01 §7 promised this: `a + b` is not an operator the compiler special-cases — it is sugar for a trait method call. `a + b` desugars to `Add::add(a, b)`; `==` to `PartialEq::eq`; `<` to `PartialOrd::lt`; `std::fmt::Display` answers `{}`; `std::fmt::Debug` answers `{:?}`. Implementing the trait is *how you define the operator* — and the orphan rule (§10.2) is what keeps the semantics sane.

### `Add`, with an owned receiver


In [ ]:
// 10.1 Add for Qty: `a + b` is Add::add(a, b); operands are consumed.
use std::ops::{Add, Sub, Mul};
impl Add for Qty {
    type Output = Qty;
    fn add(self, rhs: Qty) -> Qty {
        Qty(self.0 + rhs.0)
    }
}
impl Sub for Qty {
    type Output = Qty;
    fn sub(self, rhs: Qty) -> Qty {
        Qty(self.0 - rhs.0)
    }
}
/// Mul by a bare u32: one operand generic, one concrete.
impl Mul<u32> for Qty {
    type Output = Qty;
    fn mul(self, k: u32) -> Qty {
        Qty(self.0 * k)
    }
}
let q = Qty(300);
let sum = q + Qty(20);
let scaled = q * 3;
println!("{q:?} + {sum:?}, x3 = {scaled:?}");
assert_eq!(sum, Qty(320));
assert_eq!(scaled, Qty(900));
assert_eq!(q - Qty(100), Qty(200));


Three facts in that cell, each a deliberate surprise:

- **The receiver is `self` by value.** `a + b` *consumes both operands* — `println!("{q:?}")` after `q + Qty(20)` printed fine only because `Qty` is `Copy`. With a non-`Copy` type the first `+` moves the left operand, exactly like §2's `into_parts`. This is the ownership rule chapter 05 taught, now operating inside operator syntax.
- **`type Output`** — the trait's **associated type** — names what `+` produces: `Qty + Qty = Qty` here, `Qty * u32 = Qty` there. One trait, many right-hand types: `Add<Rhs = Self>` is the default, and `Mul<u32>` specialises it.
- **`String + &str` works and `&str + &str` does not** because the standard library impl'd `Add<&str> for String` — and *only* there. The left operand must own the buffer; `s + "CD"` consumes `s` and hands back the `String`. Two `&str`s concatenate with `format!` instead.

### `Display` by hand — and `f.pad`


In [ ]:
// 10.2 Display by hand; a plain write! ignores {:>width} flags (f.pad fixes that).
// (std::fmt::{self, Display} already in scope from §5.1)
struct Pct(f64);
impl Display for Pct {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "{:.1}%", self.0 * 100.0)
    }
}
let p = Pct(0.125);
println!("plain: {p}");
assert_eq!(format!("{p}"), "12.5%");
// Width flags are *ignored* by a write!-based impl:
// format!("{p:>10}") is still "12.5%" — the impl never looked at f's flags.
println!("wide (ignored): [{p:>10}]");


A `write!`-based `Display` ignores the caller's width flags — `{:>10}` asks the impl to pad, and this impl never touches `f`'s settings. The fix is `f.pad`, which reads the formatter's width/alignment/fill and applies them:

### `f.pad`: the formatter-aware form


In [ ]:
// 10.3 f.pad reads the caller's width/alignment flags.
struct PadPct(f64);
impl Display for PadPct {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        f.pad(&format!("{:.1}%", self.0 * 100.0))
    }
}
let p = PadPct(0.125);
assert_eq!(format!("{p}"), "12.5%");
assert_eq!(format!("{p:>10}"), "     12.5%");
assert_eq!(format!("{p:<10}"), "12.5%     ");
println!("right [{p:>10}] left [{p:<10}]");


And the derived twin: `#[derive(Debug)]` writes `{:?}` for you (§4), but hand-written `Debug` is the same trait shape as `Display` with `{:?}` on the call side. **`Display` also buys `to_string()` for free** — the blanket impl `impl<T: Display> ToString for T` turns `p.to_string()` into an ordinary method call. The orphan rule decides *who may* write these impls:

### The orphan rule


In [ ]:
// 10.4 Orphan rule: the impl is legal only if your crate owns the trait OR the type.
// impl Display for Vec<String> { .. } is refused:
//
// error[E0117]: only traits defined in the current crate can be implemented
//               for types defined outside of the crate
//
// Legal alternatives, in order of preference:
//   1. your own type          -> impl Display for Qty          (§5.1 did this)
//   2. a newtype around the foreign type
#[derive(Debug)]
struct LotsDisplay(Vec<u32>);
impl Display for LotsDisplay {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "[{} lots]", self.0.len())
    }
}
let lots = LotsDisplay(vec![100, 200]);
println!("{lots}");
assert_eq!(format!("{lots}"), "[2 lots]");


The **orphan rule** (E0117) says: an `impl` may exist only if the trait or the type is local to your crate. It is what makes ecosystem code trustworthy — two crates can never both implement `Display for Vec<String>` and fight over which wins. When you own neither side, the **newtype wrapper** (option 2, used above) is the standard escape hatch: wrap, impl, and expose the wrapper instead. This closes the loop on §5: newtypes are simultaneously a units tool, an invariant tool, and the orphan rule's official workaround.

## 11. Static vs Dynamic Dispatch: The Cost Model

Two ways to call through a trait, and the difference is *when* the implementation is chosen:

| | **static dispatch** | **dynamic dispatch** |
|---|---|---|
| syntax | generics/`impl Trait`: `fn f(x: &impl Summarize)` | trait objects: `fn f(x: &dyn Summarize)` |
| mechanism | **monomorphisation**: a copy per concrete type | **vtable**: indirect call per element |
| code size | one copy per type used | one copy total |
| optimisation | inlining, per-type specialisation | blocked by the indirect call |
| run-time choice of type | impossible | yes — mixed vectors, plugins |

The chapter's earlier cells already used both: §7's `loudest` was static (one monomorphised copy per concrete slice type); §8's mixed loop was dynamic (one call site, vtable picks per element). The question is cost, and the probe measured it:

### The measurement, and what it really shows


In [ ]:
// 11.1 Static vs dyn dispatch over 2M calls: the probe's numbers, live.
use std::time::Instant;
trait Shape {
    fn area(&self) -> f64;
}
#[derive(Debug)]
struct Rec {
    w: f64,
    h: f64,
}
impl Shape for Rec {
    fn area(&self) -> f64 {
        self.w * self.h
    }
}
fn static_total(shapes: &[Rec]) -> f64 {
    shapes.iter().map(|s| s.area()).sum()
}
fn dyn_total(shapes: &[Box<dyn Shape>]) -> f64 {
    shapes.iter().map(|s| s.area()).sum()
}
const N: usize = 2_000_000;
let recs: Vec<Rec> = (0..N).map(|i| Rec { w: i as f64, h: 2.0 }).collect();
let boxed: Vec<Box<dyn Shape>> =
    recs.iter().map(|r| Box::new(Rec { w: r.w, h: r.h }) as Box<dyn Shape>).collect();
let t0 = Instant::now();
let s = static_total(&recs);
let st = t0.elapsed();
let t0 = Instant::now();
let d = dyn_total(&boxed);
let dt = t0.elapsed();
println!("static {st:?}, dyn {dt:?}");
assert_eq!(s, d);
let ratio = dt.as_nanos() as f64 / st.as_nanos().max(1) as f64;
println!("dyn/static ≈ {ratio:.2}x");


The numbers wander run to run — probes on this machine ranged from ≈0.9× to ≈2× — and the wander is itself the finding: **a well-predicted indirect branch is nearly free**. The costs that bite in real code are structural, not per-call: `dyn` blocks **inlining** (a body as small as `area` costs about as much as the call to it), each `Box` is a heap allocation the static twin never pays, and per-type optimisation is off the table. The rule of thumb survives the measurements: default to generics; reach for `dyn` when you genuinely need mixed types or a run-time choice, and measure if the loop is hot.

## 12. Layout: Padding, Field Order and `repr`

The compiler lays out structs however it likes — **`repr(Rust)`** — and it uses that freedom to **reorder fields** so that alignment padding is minimised. Padding still exists as a concept: every field sits at its type's alignment, and gaps appear where a small field precedes a larger one. The compiler just closes them by rearranging. Opt into **`repr(C)`** and that freedom vanishes — declaration order is frozen, gaps and all:

### repr(Rust): the compiler reorders


In [ ]:
// 12.1 Same fields, two written orders — repr(Rust) packs both to 8 bytes.
struct Padded {
    flag: u8,  // 1 byte + 3 padding
    qty: u32,  // 4 bytes
    done: u8,  // 1 byte + tail padding to align the struct
}
struct Tight {
    qty: u32,  // 4
    flag: u8,  // 1
    done: u8,  // 1 + 2 tail
}
let p = Padded { flag: 1, qty: 300, done: 2 };
let t = Tight { qty: 300, flag: 1, done: 2 };
println!(
    "Padded {} B (flag {}, qty {}, done {}), Tight {} B (qty {}, flag {}, done {})",
    size_of::<Padded>(), p.flag, p.qty, p.done,
    size_of::<Tight>(), t.qty, t.flag, t.done
);
assert_eq!(size_of::<Padded>(), 8); // compiler reordered: flag+done packed together
assert_eq!(size_of::<Tight>(), 8);  // the order you wrote is already packed
// The chapter's zero-cost claims, re-measured:
assert_eq!(size_of::<Qty>(), size_of::<u32>()); // newtype
assert_eq!(size_of::<Closed>(), 0);             // unit struct
assert_eq!(size_of::<&dyn Report>(), 16);       // fat pointer
assert_eq!(size_of::<&Rec>(), 8);               // thin pointer
println!("zero-cost facts hold");


Both are 8 bytes: `repr(Rust)` rearranged `Padded`'s fields to pack `flag` and `done` together, erasing the gap a small-before-large order would leave. You write fields in the order that reads best; the compiler owns the physical layout. Chapter 08's const-assert layout contracts still work — but what they pin under `repr(Rust)` is the *size after reordering*, not your field order.

`#[repr(C)]` is the opt-out: fields stay in declaration order with C padding rules — the choice you make when a struct crosses an FFI boundary or a wire format. Probed here: the same field set is 12 bytes under `repr(C)` versus 8 under `repr(Rust)`:

### `repr(C)`: declaration order, frozen


In [ ]:
// 12.2 repr(C): fields stay in declaration order; repr(Rust) may reorder.
#[repr(C)]
struct CPadded {
    flag: u8,
    qty: u32,
    done: u8,
}
let cp = CPadded { flag: 1, qty: 300, done: 2 };
println!("repr(C) {} bytes (qty {})", size_of::<CPadded>(), cp.qty);
assert_eq!(size_of::<CPadded>(), 12);
// repr(Rust) reordered the identical fields to 8 bytes in §12.1:
// the compiler packs flag+done together after reordering, killing one gap.


One precision the probe forced: under `repr(Rust)`, the *compiler* reorders fields to minimise padding (8 bytes in §12.1); under `repr(C)`, *you* own the order and the 12 bytes come from declaration order — 3 bytes of padding after `flag`, 3 after `done`. Neither repr is "wrong" — they answer different questions: `repr(Rust)` for pure-Rust speed, `repr(C)` for interop and layout guarantees. The large-to-small field discipline from C still pays under `repr(C)` — and reading a hex dump of a repr(C) struct is where you will *see* the padding.

## 13. Reading the Compiler: Panics and E-Codes

This chapter's diagnostics are mostly about *promises made and broken*: a constructor that misses a field, a trait bound a type does not satisfy, an impl the orphan rule forbids. Every wording below was captured from this exact toolchain.

| Diagnostic | Trigger | Message (abridged) | Fix |
|---|---|---|---|
| **E0061** | `T::h()` on a method missing its `&self` | ``this function takes 1 argument but 0 arguments were supplied`` | methods need a receiver: `T::h(&t)` or `t.h()` |
| **E0034** | two traits' same method name, dot-called | ``multiple applicable items in scope`` | fully qualified syntax (§6.2) |
| **E0038** | `dyn Trait` where Trait has a const / generic method | ``the trait `Summarize` is not dyn compatible`` | remove the const/generic, or use an enum |
| **E0063** | struct literal missing a field | ``missing field `symbol` in initializer of `Order` `` | name every field, or `..base` |
| **E0117** | foreign trait on foreign type | ``only traits defined in the current crate can be implemented for types defined outside of the crate`` | newtype (§10.4) |
| **E0161/E0507** | calling a `self`-by-value method through `&dyn` | ``cannot move out of `*d` which is behind a shared reference`` | take `&self`, or box and consume the box |
| **E0308** | `-> impl Trait` with two return types in the body | `` `if` and `else` have incompatible types `` | one hidden type only (§7) |
| **E0382** | use after `..base` partial move / after `+` moved an operand | ``borrow of moved value: `t.b` `` | clone, or reorder (§1.2, §10.1) |
| **E0605** | casting a fieldful enum to its discriminant | ``non-primitive cast: `Fill` as `u8` `` | match and return the tag explicitly |

Two worth dwelling on. **E0038's modern name is "dyn compatible"** (older materials say "object safe") — the same rule, renamed; the probe's diagnostics carry the new wording. And **E0117 is a feature, not a bureaucracy**: it is why no crate can silently change how `Vec<String>` displays under yours.

### The catalogue, verified live where runnable


In [ ]:
// 13.1 The field/method shadow and the UFCS disambiguation, re-run for the record.
let q = Quote { symbol: String::from("MSFT"), cents: 41_500 };
let via_named = <Quote as Named>::describe(&q);
let via_priced = <Quote as Priced>::describe(&q);
println!("{via_named} / {via_priced}");
assert!(via_named != via_priced);
// The E0034 diagnostic this dodges:
// q.describe(); // error[E0034]: multiple applicable items in scope


## 14. Mental Model: The Three-Way Split

| Question | Answer with | Cost |
|---|---|---|
| "What data does it hold?" | a `struct` (named / tuple / unit) | the fields, padded to alignment |
| "What can it do?" | an `impl` block | functions; receivers choose borrow/move |
| "What can it promise?" | a `trait` | nothing at runtime until called through |
| "What does it work with?" | generics + bounds | one copy per type (§11) |
| "Whose type decides at run time?" | `dyn Trait` | a fat pointer + indirect call (§11) |

Read the table as a design loop: *data* is the struct; *behaviour* is the impl; *contract* is the trait; *flexibility* is the choice between static and dynamic dispatch. Python answers all five questions with "the class"; C++ answers four of five with "the class" and templates for the fourth; Rust keeps them separate so each can be reasoned about — and paid for — independently. That separation is the "OOP" of this chapter's title: Rust has *objects* (data + methods), has *polymorphism* (traits, two dispatch strategies), and deliberately has **no inheritance** — composition via traits replaces the class hierarchy, and the compiler's E-codes above are what that trade looks like.

## 15. Cheat Sheet and Review

| Task | Code |
|---|---|
| named struct + literal | `struct O { q: u32 }` / `let o = O { q: 1 };` |
| newtype | `struct Qty(u32);` (zero-cost, distinct) |
| unit marker | `struct Closed;` (0 bytes) |
| functional update | `Order { qty: 7, ..base }` (moves non-Copy fields out of base) |
| associated fn / method | `Order::new(..)` / `o.notional()` |
| receivers | `&self` read, `&mut self` mutate, `self` consume |
| qualified call | `<Quote as Named>::describe(&q)` |
| trait + default | `fn summary(&self) -> String { .. }` in the trait |
| generic + bound | `impl<T: Copy> Candle<T>` / `where T: A + B` |
| arg-position impl Trait | `fn f(x: &impl Summarize)` |
| hidden return type | `-> impl Trait` (**one** type per body) |
| trait object | `Box<dyn Trait>`, `&dyn Trait` (fat, 16 B) |
| derive set | `#[derive(Debug, Clone, PartialEq, Eq, Hash, Default)]` |
| custom `+` | `impl Add for Qty { type Output = Qty; .. }` |
| `Display` | `impl Display for T { fn fmt(&self, f: &mut Formatter<'_>) -> fmt::Result }` |
| width-aware Display | `f.pad(&string)` instead of bare `write!` |
| discriminant | fieldless only: `Level::Mid as u8`; fieldful = E0605 |
| size facts | newtype = inner; unit = 0; `&dyn` = 16 |

### Review questions

Answer from memory; each one is a §-numbered cell away.

1. Why does `Order { qty: 7, ..o2 }` leave `o2` usable while `Order { symbol, ..o2 }` does not? What exact rule decides, and what diagnostic does the latter produce *at first use of the moved field*? (§1.2)
2. `t.notional(4126.5)` and `Ticket::notional(&t, 4126.5)` — same call. What does that reveal about what a method *is*, and what does the auto-ref rule let you omit? (§2)
3. A struct has private fields, one `new`, and accessors. Which diagnostic refuses `Account { balance_cents: 5 }` from another module, and what design property does that buy? (§3)
4. `Eq` derives no methods at all. What is it *for*, and why do `HashMap` keys demand `Eq` in addition to `PartialEq`? (§4)
5. A trait with an associated const cannot be used as `dyn Trait`. Which E-code, what is the vtable-level reason, and what are the two standard fixes? (§6, §8, §13)
6. Why is `Candle::<String>::range()` a compile error while `Candle::<f64>::range()` compiles? At what *granularity* do bounds apply? (§7)
7. Your function `-> impl Summarize` returns `Trade` on one branch and `Halt` on the other. Why is this refused, what does the compiler actually report, and what are the two ways out? (§7, §8)
8. `a + b` consumed `a`. Which trait method ran, what was its receiver, and how does `Qty`'s `Copy` derive make §10.1's print legal? (§10)
9. `String + &str` compiles; `&str + &str` does not. Which impl is doing the work, and what does that impl do to its left operand? (§10)
10. repr(Rust) packed `Padded` and `Tight` to the same 8 bytes; repr(C) kept 12. Who owns the field order under each repr, where did the 4 missing bytes go, and when would you deliberately freeze the order? (§12)

### What Comes Next

Chapter 10 built the types; the applied track now teaches what to do when they fail. The next three chapters turn this chapter's raw material into working Rust services:

| Chapter | Title | What it settles |
|---|---|---|
| **11** | **Errors in Depth** | error enums for the `Option`s this chapter's constructors returned, `From`/`?`, `Box<dyn Error>` — the runtime twin of `dyn Summarize` |
| 12 | Iterators in Depth | the `Iterator` trait from the author's side — the trait every §7 generic secretly uses |
| 13 | Modules, Crates and Testing | where `pub` finally bites: real module trees, and tests for every type this chapter built |

**Carry forward.** You should now be able to do these without looking anything up:

- build a type from the three struct forms and choose constructors over literal exposure
- attach behaviour with impl blocks and pick the right receiver (`&self`/`&mut self`/`self`)
- write a trait with a default method and implement it for several types
- make a type generic with bounds, and say when you need `dyn` instead
- implement `Add`/`Display` for a newtype and explain what the orphan rule protects
- recite the dispatch cost model, and say who owns field order under each repr
